In [1]:
import os
import pickle
from pathlib import Path

import pandas as pd
import plotly.express as px
import requests
from pyalex import Works

%load_ext autoreload
%autoreload 2
input_query_openalex = """
("risk assessment" OR "risk analysis") AND
("Offshore windfarm" OR "Offshore windmill" OR "Offshore wind turbine" OR "Offshore wind park" OR "OWF") AND
("Strategic" OR "Static" OR "Long-term" OR "Marine spatial planning" OR "Large scale") AND
("maritime" OR  "navigational" OR "nautical")
"""
out_file_openalex = Path("..") / "tests" / "data" / "01_test_search_openalex.pickle"
out_file_openalex.parent.mkdir(parents=True, exist_ok=True)
work_openalex = Works().search_filter(title_and_abstract=input_query_openalex)
with open(out_file_openalex, "wb") as f:
    pickle.dump(work_openalex, f)

In [2]:
from slrrag.querier_openalex import work_openalex_to_list

l_test_openalex = work_openalex_to_list(work_openalex)
out_file_openalex_list = (
    Path("..") / "tests" / "data" / "01_test_search_openalex_list.pickle"
)
print(out_file_openalex_list)
with open(out_file_openalex_list, "wb") as f:
    pickle.dump(l_test_openalex, f)
l_test_openalex

5 found in Openalex.


  0%|          | 0/0.1 [00:00<?, ?it/s]


RetryError: HTTPSConnectionPool(host='api.openalex.org', port=443): Max retries exceeded with url: /works?filter=title_and_abstract.search:%0A%28%22risk+assessment%22+OR+%22risk+analysis%22%29+AND%0A%28%22Offshore+windfarm%22+OR+%22Offshore+windmill%22+OR+%22Offshore+wind+turbine%22+OR+%22Offshore+wind+park%22+OR+%22OWF%22%29+AND%0A%28%22Strategic%22+OR+%22Static%22+OR+%22Long-term%22+OR+%22Marine+spatial+planning%22+OR+%22Large+scale%22%29+AND%0A%28%22maritime%22+OR++%22navigational%22+OR+%22nautical%22%29%0A&per-page=50&cursor=%2A (Caused by ResponseError('too many 429 error responses'))

In [3]:
from slrrag.querier_openalex import query_openalex_and_standardize

query_openalex_and_standardize(input_query_openalex, n_paper_max=1)

ImportError: cannot import name 'query_openalex_and_standardize' from 'slrrag.querier_openalex' (/home/vlitoux/Documents/SLR-RAG/src/slrrag/querier_openalex.py)

query

In [13]:
import pybliometrics
from pybliometrics.scopus import ScopusSearch

pybliometrics.init()  # Init for scopus
input_query_test_scopus = """
((("offshore") AND ("infrastructure" OR "construction" OR "installation" OR "windfarm" OR "wind farm" OR "windmill" OR "wind turbine" OR "wind park" OR "wind energy" OR "platform" OR "quay" OR "wharf" OR "lidar")) OR ("bridge" OR "flotel" OR "Aquaculture farm" OR "Artificial Island" OR "Synthetic Island")) AND
(("Navigation risk assessment" OR "Navigational risk assessment" OR "Navigation risk assessments" OR "Navigational risk assessments"))
"""
input_query_test_scopus = f"""
TITLE-ABS-KEY( {input_query_test_scopus})
"""
scopus_search = ScopusSearch(
    query=input_query_test_scopus, verbose=True, subscriber=True, sort="+relevancy"
)
scopus_search.get_results_size()

16

In [14]:
from slrrag.querier_scopus import scopus_to_dict_list
df_scopus = scopus_to_dict_list(scopus_search)
df_scopus

,doi,title,date,abstract,id_scopus,type,year,keywords_scopus,relevance_order_scopus
0,10.1088/1755-1315/1585/1/012044,Risk Based Decision Making for Decommissioning...,2026-01-01,Decommissioning offshore platforms demands tho...,2-s2.0-105033024716,conference-paper,<NA>,NaN,0
1,10.3390/jmse13101863,Dijkstra and A* Algorithms for Algorithmic Opt...,2025-10-01,Shipping in complex marine environments requir...,2-s2.0-105020158469,article,<NA>,A* algorithm | Dijkstra algorithm | maritime l...,1
2,10.3390/app15126943,Navigational Risk Assessment in Offshore Wind ...,2025-06-01,Navigation in offshore wind farm (OWF) areas i...,2-s2.0-105008974623,article,<NA>,hydrometeorological impact on navigational saf...,2
3,10.1061/AJRUA6.RUENG-1477,Incorporation of Fault Tree Analysis and Evide...,2025-06-01,Quantifying the navigation risk in waters of o...,2-s2.0-105000646497,article,<NA>,Evidential reasoning | Fault tree analysis | N...,3
4,10.3969/j.issn.1001-0505.2024.03.022,Analysis on navigation risk of ships in bridge...,2024-05-01,To assess the navigation risk of ships in the ...,2-s2.0-85195450739,article,<NA>,bridge areas | flow characteristics | hydrodyn...,4
5,10.1016/j.ocecoaman.2022.106428,A novel fuzzy multi-factor navigational risk a...,2023-02-01,The navigational risk assessment in the waters...,2-s2.0-85142526670,article,<NA>,Analytic hierarchy process | Fuzzy comprehensi...,5
6,10.1088/1742-6596/2618/1/012005,Multi-functional and Practical Adaptive Collis...,2023-01-01,"In the current landscape, autonomous navigatio...",2-s2.0-85179626005,conference-paper,<NA>,NaN,6
7,10.11990/jheu.202008049,Navigation risk assessment in bridge areas bas...,2022-05-05,To solve the navigation safety problems in bri...,2-s2.0-85131226808,article,<NA>,Bridge area | Fuzzy synthetic evaluation metho...,7
8,10.3390/jmse10050659,Utilization of Full-Mission Ship-Handling Simu...,2022-05-01,"The MV Ever Given accident, which took place i...",2-s2.0-85130315916,article,<NA>,Istanbul Strait | large vessel | maritime traf...,8
9,10.1016/j.ocecoaman.2022.106078,Assessing the validity of navigation risk asse...,2022-03-15,The developments of offshore wind farms can pl...,2-s2.0-85124249267,article,<NA>,Maritime risk assessment | Navigation risk ass...,9


In [15]:
from pybliometrics.scopus import ScopusSearch, AbstractRetrieval

# For a single document
def complete_scopus_infos(id_paper : str):
    doc = AbstractRetrieval(identifier=id_paper, view="FULL")
    area_text = " | ".join([x.area for x in doc.subject_areas])
    return pd.Series((doc.language,area_text))

df_scopus[["language", "subject_area" ]] = df_scopus["id_scopus"].apply(complete_scopus_infos)
df_scopus

,doi,title,date,abstract,id_scopus,type,year,keywords_scopus,relevance_order_scopus,language,subject_area
0,10.1088/1755-1315/1585/1/012044,Risk Based Decision Making for Decommissioning...,2026-01-01,Decommissioning offshore platforms demands tho...,2-s2.0-105033024716,conference-paper,<NA>,NaN,0,eng,Environmental Science (all) | Earth and Planet...
1,10.3390/jmse13101863,Dijkstra and A* Algorithms for Algorithmic Opt...,2025-10-01,Shipping in complex marine environments requir...,2-s2.0-105020158469,article,<NA>,A* algorithm | Dijkstra algorithm | maritime l...,1,eng,Civil and Structural Engineering | Water Scien...
2,10.3390/app15126943,Navigational Risk Assessment in Offshore Wind ...,2025-06-01,Navigation in offshore wind farm (OWF) areas i...,2-s2.0-105008974623,article,<NA>,hydrometeorological impact on navigational saf...,2,eng,Materials Science (all) | Instrumentation | En...
3,10.1061/AJRUA6.RUENG-1477,Incorporation of Fault Tree Analysis and Evide...,2025-06-01,Quantifying the navigation risk in waters of o...,2-s2.0-105000646497,article,<NA>,Evidential reasoning | Fault tree analysis | N...,3,eng,Civil and Structural Engineering | Building an...
4,10.3969/j.issn.1001-0505.2024.03.022,Analysis on navigation risk of ships in bridge...,2024-05-01,To assess the navigation risk of ships in the ...,2-s2.0-85195450739,article,<NA>,bridge areas | flow characteristics | hydrodyn...,4,chi,Engineering (all)
5,10.1016/j.ocecoaman.2022.106428,A novel fuzzy multi-factor navigational risk a...,2023-02-01,The navigational risk assessment in the waters...,2-s2.0-85142526670,article,<NA>,Analytic hierarchy process | Fuzzy comprehensi...,5,eng,"Oceanography | Aquatic Science | Management, M..."
6,10.1088/1742-6596/2618/1/012005,Multi-functional and Practical Adaptive Collis...,2023-01-01,"In the current landscape, autonomous navigatio...",2-s2.0-85179626005,conference-paper,<NA>,NaN,6,eng,Physics and Astronomy (all)
7,10.11990/jheu.202008049,Navigation risk assessment in bridge areas bas...,2022-05-05,To solve the navigation safety problems in bri...,2-s2.0-85131226808,article,<NA>,Bridge area | Fuzzy synthetic evaluation metho...,7,chi,Control and Systems Engineering | Chemical Eng...
8,10.3390/jmse10050659,Utilization of Full-Mission Ship-Handling Simu...,2022-05-01,"The MV Ever Given accident, which took place i...",2-s2.0-85130315916,article,<NA>,Istanbul Strait | large vessel | maritime traf...,8,eng,Civil and Structural Engineering | Water Scien...
9,10.1016/j.ocecoaman.2022.106078,Assessing the validity of navigation risk asse...,2022-03-15,The developments of offshore wind farms can pl...,2-s2.0-85124249267,article,<NA>,Maritime risk assessment | Navigation risk ass...,9,eng,"Oceanography | Aquatic Science | Management, M..."


In [16]:
df_scopus[["language", "subject_area" ]]

,language,subject_area
0,eng,Environmental Science (all) | Earth and Planet...
1,eng,Civil and Structural Engineering | Water Scien...
2,eng,Materials Science (all) | Instrumentation | En...
3,eng,Civil and Structural Engineering | Building an...
4,chi,Engineering (all)
5,eng,"Oceanography | Aquatic Science | Management, M..."
6,eng,Physics and Astronomy (all)
7,chi,Control and Systems Engineering | Chemical Eng...
8,eng,Civil and Structural Engineering | Water Scien...
9,eng,"Oceanography | Aquatic Science | Management, M..."


In [42]:
df_scopus["id_scopus"].apply(complete_scopus_infos)

0     (eng, [(Environmental Science (all), ENVI, 230...
1     (eng, [(Civil and Structural Engineering, ENGI...
2     (eng, [(Materials Science (all), MATE, 2500), ...
3     (eng, [(Civil and Structural Engineering, ENGI...
4              (chi, [(Engineering (all), ENGI, 2200)])
5     (eng, [(Oceanography, EART, 1910), (Aquatic Sc...
6     (eng, [(Physics and Astronomy (all), PHYS, 310...
7     (chi, [(Control and Systems Engineering, ENGI,...
8     (eng, [(Civil and Structural Engineering, ENGI...
9     (eng, [(Oceanography, EART, 1910), (Aquatic Sc...
10    (eng, [(Environmental Science (all), ENVI, 230...
11             (chi, [(Engineering (all), ENGI, 2200)])
12    (eng, [(Mathematics (all), MATH, 2600), (Engin...
13    (eng, [(Human Factors and Ergonomics, SOCI, 33...
14             (eng, [(Ocean Engineering, ENGI, 2212)])
15    (eng, [(Safety, Risk, Reliability and Quality,...
Name: id_scopus, dtype: object

In [24]:
import pickle
from pathlib import Path

out_file_scopus = Path("..") / "tests" / "data" / "01_test_search_scopus.pickle"
with open(out_file_scopus, "wb") as f:
    pickle.dump(scopus_search, f)

In [ ]:
with open(out_file_scopus, "rb") as f:
    t = pickle.load(f)
t.results

[Document(eid='2-s2.0-105033024716', doi='10.1088/1755-1315/1585/1/012044', pii=None, pubmed_id=None, title='Risk Based Decision Making for Decommissioning of Fixed Jacket Platforms', subtype='cp', subtypeDescription='Conference Paper', creator='Silvianita ', afid='60070707;60197135', affilname='Institut Teknologi Sepuluh Nopember;College of Engineering,\xa0Qatar University', affiliation_city='Surabaya;Doha', affiliation_country='Indonesia;Qatar', author_count='4', author_names='Silvianita, ;Wibowo, A. C.;Satrio, D.;Khamidi, M. F.', author_ids='36675978100;59157007100;57202198993;27567869500', author_afids='60070707;60070707;60070707;60197135', coverDate='2026-01-01', coverDisplayDate='2026', publicationName='Iop Conference Series Earth and Environmental Science', issn='17551307', source_id='19900195068', eIssn='17551315', aggregationType='Conference Proceeding', volume='1585', issueIdentifier='1', article_number='012044', pageRange=None, description='Decommissioning offshore platforms

In [12]:
t

In [ ]:
import pybliometrics
from pybliometrics.scopus import ScopusSearch

pybliometrics.init()  # Init for scopus
scopus_search = ScopusSearch(
    query='title("Navigational Risk assessment")',
    verbose=True,
    subscriber=False,
    sort="-relevance",
    refresh=True,
)
scopus_search

In [ ]:
data = [
    {
        "doi": paper.doi,
        "title": paper.title,
        "date": paper.coverDate,
        "abstract": paper.description,
        "id_scopus": paper.eid,
        "type": paper.subtypeDescription,
        "keywords_scopus": paper.authkeywords,
    }
    for paper in scopus_search.results
]
pd.DataFrame(data).assign(
    "relevace_oreder",
)

In [ ]:
[x.title for x in scopus_search.results]

In [ ]:
scopus_api_key = os.environ.get("SCOPUS_API_KEY")
assert scopus_api_key is not None
url = "https://api.elsevier.com/content/search/scopus"
params = {"query": 'title("navigational risk assessment")', "apiKey": scopus_api_key}
headers = {"Accept": "application/json"}

response = requests.get(url, headers=headers, params=params)

In [ ]:
dict_an = response.json()
dict_an

In [ ]:
dict_an["search-results"].keys()

In [ ]:
import pandas as pd

df_all = pd.DataFrame(dict_an["search-results"]["entry"])
df_all.iloc[0].T

In [ ]:
[x.authkeywords for x in scopus_search.results]

In [ ]:
fig = px.bar(
    l_test_openalex.groupby("topic_openalex")
    .size()
    .sort_values(ascending=False)
    .head(10)
    .rename("count")
    .reset_index(),
    y="topic_openalex",
    x="count",
)
# fig.update_xaxes(tickangle=90)
fig